In [1]:
from pymongo import MongoClient
import re
import os
from typing import List, Dict, Any
import shutil

In [2]:
def connect_mongodb(uri,db_name, collection_name):
    # Connect to MongoDB 
    client = MongoClient(uri)
    
    # Select the database and collection
    db = client[db_name]
    collection = db[collection_name]
    
    return collection
def query_mongodb(collection, query, fields=None):
    """
    Queries the MongoDB collection and returns specific fields or all fields if no specific ones are provided.
    
    Args:
        collection: The MongoDB collection object.
        query (dict): The query to filter documents.
        fields (list): List of fields to return from the query. If None, all fields are returned.
    
    Returns:
        result: List of documents with the specified fields or all fields.
    """
    # If fields is None, return all fields
    if fields:
        # Return only the specified fields
        projection = {field: 1 for field in fields}
    else:
        # Return all fields
        projection = None
    
    # Perform the query
    result = collection.find(query, projection)
    
    # Convert the cursor to a list and return
    return list(result)

def create_directory_if_not_exists(directory_path):
    if not os.path.exists(directory_path):  # Check if the directory exists
        try:
            os.makedirs(directory_path)  # Create the directory
            return f"Directory '{directory_path}' has been created."
        except Exception as e:
            return f"Error creating directory: {e}"
    else:
        return f"Directory '{directory_path}' already exists."
# Function search file path that contain keywork
def find_files_with_keyword(directory, keyword):
    matching_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            file_path = os.path.join(root, file)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                    if keyword in content:
                        matching_files.append(file_path)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return matching_files
def get_third_party_keyword_data(collection) -> List[Dict[str, Any]]:
    """
    Retrieve documents containing non-empty 'third-party-keyword'.

    Args:
        collection (pymongo.collection.Collection): The MongoDB collection to query.

    Returns:
        List[Dict[str, Any]]: A list of documents containing '_id' and 'third-party-keyword'.
    """
    query = {
        "third-party-keyword": {"$exists": True, "$ne": {}}
    }
    projection = {
        "_id": 1,
        "third-party-keyword": 1
    }

    results = collection.find(query, projection)
    
    # Collect all matching documents
    data = []
    for doc in results:
        if isinstance(doc.get("third-party-keyword"), dict) and any(doc["third-party-keyword"].values()):
            data.append({
                "_id": doc["_id"],
                "third-party-keyword": doc["third-party-keyword"]
            })
    
    return data
def extract_keywords(data: Dict[str, List[str]]) -> List[str]:
    """
    Extract keywords from the given dictionary and return them as a list.

    Args:
        data (Dict[str, List[str]]): The dictionary containing keywords as keys.

    Returns:
        List[str]: A list of keywords from the dictionary.
    """
    return list(data.keys())
def extract_import_lib(import_string: str) -> str:
    """
    Extracts the X part from an import statement of the form: 'import X'.

    Args:
        import_string (str): The import statement string.

    Returns:
        str: The extracted X part.
    """
    pattern = r'^import\s+(.+)$'
    match = re.match(pattern, import_string)
    if match:
        return match.group(1)
    else:
        return None
def copy_file(file_path: str, destination_dir: str) -> None:
    """
    Copies a file to the specified destination directory. If a duplicate is found,
    it automatically renames the file by appending '-duplicate-N' before the extension.

    Args:
        file_path (str): The path of the file to be copied.
        destination_dir (str): The directory where the file will be copied to.

    Returns:
        None
    """
    # Check if the source file exists
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"The file '{file_path}' does not exist.")
    
    # Check if the destination directory exists, if not, create it
    if not os.path.exists(destination_dir):
        os.makedirs(destination_dir)
    
    # Extract the file name and extension from the file path
    file_name, file_extension = os.path.splitext(os.path.basename(file_path))
    
    # Generate the initial destination path
    destination_file_path = os.path.join(destination_dir, f"{file_name}{file_extension}")
    
    duplicate_count = 1
    
    # Automatically rename the file if it already exists
    while os.path.exists(destination_file_path):
        #print(f"Duplicate file found: '{destination_file_path}'")
        destination_file_path = os.path.join(destination_dir, f"{file_name}-duplicate-{duplicate_count}{file_extension}")
        duplicate_count += 1
    
    # Copy the file to the destination directory
    shutil.copy(file_path, destination_file_path)
    #print(f"File '{file_path}' successfully copied to '{destination_file_path}'.")


In [3]:
# mongoDB_uri = 'mongodb://192.168.1.14:27017'
# mongoDB_database = 'wearable-project' 
# mongoDB_collection = 'wearable-standalone'
# local
mongoDB_uri = 'mongodb://localhost:27017'
mongoDB_database = 'wearable-project' 
# stand-alone collection
# mongoDB_collection = 'wearable-standalone-02-04-2025'
# root_directory = r'E:\wearable-standalone-code-extraction'
# source_code_root_directory = r"E:\wearable-project-decompile-source\source-code-standalone"
# app collection
mongoDB_collection = 'wearable-app-02-04-2025'
root_directory = r'E:\wearable-app-code-extraction'
source_code_root_directory = r"E:\wearable-project-decompile-source\source-code-wearable"

In [4]:
# Connect to the MongoDB collection
collection = connect_mongodb(mongoDB_uri,mongoDB_database,mongoDB_collection)
# Fetch the specified fields only
result_data = get_third_party_keyword_data(collection)
print(len(result_data))
# print(result)

4714


In [5]:
for i in range(192, len(result_data)):
    print("------------------ Loop-"+str(i)+"------------------")
    data = result_data[i]
    app_id = data["_id"]
    print("app_id: ", app_id)
    # app_directory = source_code_root_directory+"\\"+app_id+"-java"
    app_directory = source_code_root_directory+"\\"+app_id+".apk"
    extract_code_directory = root_directory+"\\"+app_id
    create_directory_if_not_exists(extract_code_directory)
    # print("app_directory: ",app_directory)
    app_third_party = data["third-party-keyword"]
    # print("app_third_party_keyword: ",app_third_party)
    third_party_name_arr = extract_keywords(app_third_party)
    # print("third_party_name_arr: ",third_party_name_arr)
    for j in range(len(third_party_name_arr)):
        third_party_name = third_party_name_arr[j]
        third_party_directory = extract_code_directory+"\\"+third_party_name
        create_directory_if_not_exists(third_party_directory)
        # print("*********** Third-party: "+third_party_name+" ***********")
        # print("third_party_name: ",third_party_name)
        # print("third_party_directory :",third_party_directory)
        third_party_keyword_arr = app_third_party[third_party_name]
        # print("third_party_keyword_arr: ",third_party_keyword_arr)
        for x in range(len(third_party_keyword_arr)):
            keyword = third_party_keyword_arr[x]
            # print("==== Third-party-keyword: "+keyword+" ====")
            matching_files = find_files_with_keyword(app_directory, keyword)
            # print("We found "+ str(len(matching_files)) + " file with keyword")
            if(len(matching_files)>0):
                code_file_directory = third_party_directory+"\\"+extract_import_lib(keyword)
                create_directory_if_not_exists(code_file_directory)
                for y in range(len(matching_files)):
                    file_path = matching_files[y]
                    copy_file(file_path, code_file_directory)
    #     break
    # break

------------------ Loop-192------------------
app_id:  de.stefanheintz.smarthome


KeyboardInterrupt: 

Number of file contain keyword com.clevertap is: 152
Files containing the keyword com.clevertap
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\a24me\groupcal\fcm\FCMReceiver.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\a24me\groupcal\managers\a.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\a24me\groupcal\managers\q5.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\com\clevertap\android\sdk\a.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\com\clevertap\android\sdk\b.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\com\clevertap\android\sdk\c.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\com\clevertap\android\sdk\CleverTapInstanceConfig.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-

Number of file contain keyword com.zendesk is: 3
Files containing the keyword com.zendesk
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\com\zendesk\service\ZendeskDateTypeAdapter.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\zendesk\belvedere\v.java
E:\wearable-project-decompile-source\source-code-standalone\app.groupcal.www-java\zendesk\core\ZendeskApplicationModule.java
